In [ ]:
import os
import numpy as np
import pandas as pd
import plotly.express as px

import utils
import billing_calcs

from const import *
from datetime import datetime, time
from sklearn.decomposition import PCA, NMF, FastICA
from sklearn.cluster import KMeans

In [ ]:
half_hourly_usage_df = utils.get_half_hourly_usage_data()
monthly_usage_df = utils.get_monthly_usage_data()

display(half_hourly_usage_df.head(), f"{len(half_hourly_usage_df)} rows")
display(monthly_usage_df.head(), f"{len(monthly_usage_df)} rows")

In [ ]:
# condense overlapping DST hours
duplicated_idx = half_hourly_usage_df[START_DT_COL].duplicated(keep='first')
adjustment_df = half_hourly_usage_df.loc[duplicated_idx].groupby(START_DT_COL)[UNITS_COL].sum()
half_hourly_usage_df = half_hourly_usage_df.loc[~duplicated_idx]
update_idx = half_hourly_usage_df[START_DT_COL].isin(adjustment_df.index)
half_hourly_usage_df.loc[update_idx, UNITS_COL] = half_hourly_usage_df.loc[update_idx, UNITS_COL] + adjustment_df

In [ ]:
half_hourly_usage_df['time'] = half_hourly_usage_df[START_DT_COL].dt.time
half_hourly_usage_df['day'] = half_hourly_usage_df[START_DT_COL].dt.day_name()
half_hourly_usage_df['date'] = half_hourly_usage_df[START_DT_COL].dt.date

In [ ]:
half_hourly_usage_df['units'].interpolate

In [ ]:
weekday_usage_df = (
    half_hourly_usage_df[half_hourly_usage_df['day'].isin(WEEKDAYS)]
    .pivot(index=['date','day'], columns='time', values=UNITS_COL)
)
weekday_usage_df.fillna(weekday_usage_df.mean().mean(), inplace=True)
display(weekday_usage_df)
X_weekday = weekday_usage_df.to_numpy()

print(billing_calcs.calc_simple_bill(weekday_usage_df, fixed_rate=1.5, variable_rate=0.2749))

print(
    billing_calcs.calc_contact_good_charge_bill(
        weekday_usage_df.copy()
    )
)

In [ ]:
os.environ["OMP_NUM_THREADS"] = '3'
n_components = 10
weekday_kmeans = KMeans(n_clusters=n_components).fit(X_weekday)
H_weekday = weekday_kmeans.cluster_centers_
W_weekday = X_weekday@np.linalg.pinv(H_weekday) 

fig = px.imshow(H_weekday)
fig.update_layout(
    height=800,
    width=800,
)
fig.show()

df = pd.DataFrame(H_weekday.T, index=weekday_usage_df.columns)
fig = px.line(df,y=list(range(n_components)))
fig.show()

In [ ]:
n_components=5
nmf_est = NMF(
    n_components=n_components,
    solver='mu',
    beta_loss='kullback-leibler',
    # beta_loss='itakura-saito',
    l1_ratio=1e-3,
    alpha_H=1e-3,
    max_iter=int(1e4),
).fit(X_weekday)

W_weekday = nmf_est.transform(X_weekday)
H_weekday  = nmf_est.components_

print(W_weekday.shape, H_weekday.shape)

print(f"Error: {np.linalg.norm(X_weekday - W_weekday@H_weekday)}")

fig = px.imshow(H_weekday)
fig.update_layout(
    height=800,
    width=800,
)
fig.show()

df = pd.DataFrame(H_weekday.T, index=weekend_usage_df.columns)
fig = px.line(df,y=list(range(n_components)))
fig.show()

In [ ]:
approx_weekday_usage_df = pd.DataFrame(
    W_weekday@H_weekday, 
    index=weekday_usage_df.index, 
    columns=weekday_usage_df.columns
)

print(
    billing_calcs.calc_simple_bill(
        approx_weekday_usage_df.copy(), 
        fixed_rate=1.5, 
        variable_rate=0.2749
    )
)
print(
    billing_calcs.calc_contact_good_charge_bill(
        approx_weekday_usage_df.copy()
    )
)


time_cols = approx_weekday_usage_df.columns
approx_half_hourly_weekday_usage_df = approx_weekday_usage_df.reset_index()
approx_half_hourly_weekday_usage_df = pd.melt(
    approx_half_hourly_weekday_usage_df,
    id_vars=['date'],
    value_vars=time_cols,
    value_name='units',
    var_name='time',
)

approx_half_hourly_weekday_usage_df['start_dt'] = pd.to_datetime(
    approx_half_hourly_weekday_usage_df['date'].apply(lambda x: x.strftime("%d/%m/%Y"))
    + ' ' + approx_half_hourly_weekday_usage_df['time'].apply(lambda x: x.strftime("%H:%M")),
    dayfirst=True,
)
approx_half_hourly_weekday_usage_df.sort_values('start_dt',inplace=True)
approx_half_hourly_weekday_usage_df['series'] = 'approximation'

half_hourly_usage_df['series'] = 'original'

comparison_half_hourly_usage_df = pd.concat(
    [
        approx_half_hourly_weekday_usage_df[['start_dt', 'units', 'series']],
        half_hourly_usage_df[['start_dt', 'units', 'series']]
    ]
)

diff = (
    approx_half_hourly_weekday_usage_df.set_index('start_dt')['units'] 
    - half_hourly_usage_df.set_index('start_dt')['units']
)

px.line(comparison_half_hourly_usage_df, x='start_dt', y='units', color='series').show()
px.line(diff).show()
print(diff.sum())

In [ ]:
weekend_usage_df = (
    half_hourly_usage_df[half_hourly_usage_df['day'].isin(WEEKENDS)]
    .pivot(index=['date','day'], columns='time', values=UNITS_COL)
)
weekend_usage_df.fillna(weekend_usage_df.mean().mean(), inplace=True)
display(weekend_usage_df)
X_weekend = weekend_usage_df.to_numpy()

print(billing_calcs.calc_simple_bill(weekend_usage_df, fixed_rate=1.5, variable_rate=0.2749))

print(
    billing_calcs.calc_contact_good_charge_bill(
        weekend_usage_df.copy()
    )
)

In [ ]:
n_components=5
nmf_est = NMF(
    n_components=n_components,
    solver='mu',
    beta_loss='kullback-leibler',
    # beta_loss='itakura-saito',
    l1_ratio=1e-3,
    alpha_H=1e-3,
    max_iter=int(1e4),
).fit(X_weekend)

W_weekend = nmf_est.transform(X_weekend)
H_weekend  = nmf_est.components_

print(f"Error: {np.linalg.norm(X_weekend - W_weekend@H_weekend)}")

fig = px.imshow(H_weekend)
fig.update_layout(
    height=800,
    width=800,
)
fig.show()

df = pd.DataFrame(H_weekend.T, index=weekend_usage_df.columns)
fig = px.line(df,y=list(range(n_components)))
fig.show()

In [ ]:
approx_weekend_usage_df = pd.DataFrame(
    W_weekend@H_weekend, 
    index=weekend_usage_df.index, 
    columns=weekend_usage_df.columns
)

print(
    billing_calcs.calc_simple_bill(
        approx_weekend_usage_df.copy(), 
        fixed_rate=1.5, 
        variable_rate=0.2749
    )
)
print(
    billing_calcs.calc_contact_good_charge_bill(
        approx_weekend_usage_df.copy()
    )
)


time_cols = approx_weekend_usage_df.columns
approx_half_hourly_weekend_usage_df = approx_weekend_usage_df.reset_index()
approx_half_hourly_weekend_usage_df = pd.melt(
    approx_half_hourly_weekend_usage_df,
    id_vars=['date'],
    value_vars=time_cols,
    value_name='units',
    var_name='time',
)

approx_half_hourly_weekend_usage_df['start_dt'] = pd.to_datetime(
    approx_half_hourly_weekend_usage_df['date'].apply(lambda x: x.strftime("%d/%m/%Y"))
    + ' ' + approx_half_hourly_weekend_usage_df['time'].apply(lambda x: x.strftime("%H:%M")),
    dayfirst=True,
)
approx_half_hourly_weekend_usage_df.sort_values('start_dt',inplace=True)
approx_half_hourly_weekend_usage_df['series'] = 'approximation'

half_hourly_usage_df['series'] = 'original'
comparison_half_hourly_usage_df = pd.concat(
    [
        approx_half_hourly_weekend_usage_df[['start_dt', 'units', 'series']],
        half_hourly_usage_df[['start_dt', 'units', 'series']]
    ]
)

diff = (
    approx_half_hourly_weekend_usage_df.set_index('start_dt')['units'] 
    - half_hourly_usage_df.set_index('start_dt')['units']
)

px.line(comparison_half_hourly_usage_df, x='start_dt', y='units', color='series').show()
px.line(diff).show()
print(diff.sum())